In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import pandas as pd
from copy import deepcopy

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) 
])

train_full = datasets.EMNIST(root='./data', split='balanced', train=True, 
                             download=True, transform=transform)
test_dataset = datasets.EMNIST(root='./data', split='balanced', train=False, 
                               download=True, transform=transform)

train_size = int(0.8 * len(train_full))
val_size = len(train_full) - train_size
train_dataset, val_dataset = random_split(
    train_full, [train_size, val_size], 
    generator=torch.Generator().manual_seed(SEED)
)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")
x_batch, y_batch = next(iter(train_loader))
print(f"Batch shape: x={x_batch.shape}, y={y_batch.shape}")
print(f"Value range: [{x_batch.min():.3f}, {x_batch.max():.3f}]")
print(f"Classes: {len(torch.unique(y_batch))} unique in batch, total: {len(train_full.classes)}")

class MLP(nn.Module):
    def __init__(self, input_size=28*28, hidden_sizes=[256, 128], num_classes=47, 
                 dropout_p=0.0, use_batchnorm=False):
        super(MLP, self).__init__()
        layers = [nn.Flatten()]
        prev_size = input_size
        
        for i, h_size in enumerate(hidden_sizes):
            layers.append(nn.Linear(prev_size, h_size))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h_size))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(dropout_p))
            prev_size = h_size
        
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    
    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    
    return total_loss / len(loader), correct / total


def run_experiment(config, train_ld, val_ld, test_ld, device, epochs=20, early_stop_patience=None):
    """Запуск одного эксперимента с возвратом истории и лучшей модели"""
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    
    model = MLP(
        input_size=28*28, 
        hidden_sizes=config['hidden_sizes'], 
        num_classes=47,
        dropout_p=config.get('dropout_p', 0.0),
        use_batchnorm=config.get('use_batchnorm', False)
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    
    if config['optimizer'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=config['lr'], 
                               weight_decay=config.get('weight_decay', 0.0))
    else:  
        optimizer = optim.SGD(model.parameters(), lr=config['lr'], 
                              momentum=config.get('momentum', 0.0),
                              weight_decay=config.get('weight_decay', 0.0))
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        t_loss, t_acc = train_one_epoch(model, train_ld, criterion, optimizer, device)
        v_loss, v_acc = evaluate(model, val_ld, criterion, device)
        
        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        

        if early_stop_patience is not None:
            if v_acc > best_val_acc:
                best_val_acc = v_acc
                best_model_state = deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= early_stop_patience:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
        else:
            if v_acc > best_val_acc:
                best_val_acc = v_acc
                best_model_state = deepcopy(model.state_dict())

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    test_loss, test_acc = evaluate(model, test_ld, criterion, device)
    
    return {
        'model': model,
        'history': history,
        'best_val_accuracy': best_val_acc,
        'best_val_loss': history['val_loss'][np.argmax(history['val_acc'])],
        'test_accuracy': test_acc,
        'epochs_trained': len(history['train_loss'])
    }

def save_curves(history, title, filepath):
    """Сохранение графиков обучения"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} - Loss')
    ax1.legend()
    ax1.grid(True)
    
    ax2.plot(history['train_acc'], label='Train Acc')
    ax2.plot(history['val_acc'], label='Val Acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title(f'{title} - Accuracy')
    ax2.legend()
    ax2.grid(True)
    
    plt.tight_layout()
    plt.savefig(filepath, dpi=150)
    plt.close()
    return filepath

results = []
base_config = {
    'hidden_sizes': [256, 128, 64],
    'optimizer': 'Adam',
    'lr': 1e-3,
    'dropout_p': 0.0,
    'use_batchnorm': False,
    'weight_decay': 0.0,
    'momentum': 0.0
}

print("\n=== E1: Base MLP ===")
config_e1 = base_config.copy()
result_e1 = run_experiment(config_e1, train_loader, val_loader, test_loader, device, epochs=15)
results.append({
    'experiment_id': 'E1', 'dataset': 'EMNIST', 'seed': SEED,
    'model_summary': '256-128-64/ReLU/no-Dropout/no-BN',
    'optimizer': 'Adam', 'lr': 1e-3, 'momentum': 0, 'weight_decay': 0,
    'epochs_trained': result_e1['epochs_trained'],
    'best_val_accuracy': result_e1['best_val_accuracy'],
    'best_val_loss': result_e1['best_val_loss']
})
print(f"E1 Val Acc: {result_e1['best_val_accuracy']:.4f}")

print("\n=== E2: + Dropout (p=0.3) ===")
config_e2 = base_config.copy()
config_e2['dropout_p'] = 0.3
result_e2 = run_experiment(config_e2, train_loader, val_loader, test_loader, device, epochs=15)
results.append({
    'experiment_id': 'E2', 'dataset': 'EMNIST', 'seed': SEED,
    'model_summary': '256-128-64/ReLU/Dropout(0.3)/no-BN',
    'optimizer': 'Adam', 'lr': 1e-3, 'momentum': 0, 'weight_decay': 0,
    'epochs_trained': result_e2['epochs_trained'],
    'best_val_accuracy': result_e2['best_val_accuracy'],
    'best_val_loss': result_e2['best_val_loss']
})
print(f"E2 Val Acc: {result_e2['best_val_accuracy']:.4f}")

print("\n=== E3: + BatchNorm ===")
config_e3 = base_config.copy()
config_e3['use_batchnorm'] = True
result_e3 = run_experiment(config_e3, train_loader, val_loader, test_loader, device, epochs=15)
results.append({
    'experiment_id': 'E3', 'dataset': 'EMNIST', 'seed': SEED,
    'model_summary': '256-128-64/ReLU/no-Dropout/BatchNorm',
    'optimizer': 'Adam', 'lr': 1e-3, 'momentum': 0, 'weight_decay': 0,
    'epochs_trained': result_e3['epochs_trained'],
    'best_val_accuracy': result_e3['best_val_accuracy'],
    'best_val_loss': result_e3['best_val_loss']
})
print(f"E3 Val Acc: {result_e3['best_val_accuracy']:.4f}")

print("\n=== E4: EarlyStopping ===")
best_config_a = config_e2 if result_e2['best_val_accuracy'] > result_e3['best_val_accuracy'] else config_e3
result_e4 = run_experiment(best_config_a, train_loader, val_loader, test_loader, 
                           device, epochs=30, early_stop_patience=5)
results.append({
    'experiment_id': 'E4', 'dataset': 'EMNIST', 'seed': SEED,
    'model_summary': '256-128-64/ReLU/' + ('Dropout(0.3)' if best_config_a['dropout_p']>0 else 'no-Dropout') + '/' + ('BatchNorm' if best_config_a['use_batchnorm'] else 'no-BN') + '/EarlyStop',
    'optimizer': 'Adam', 'lr': 1e-3, 'momentum': 0, 'weight_decay': 0,
    'epochs_trained': result_e4['epochs_trained'],
    'best_val_accuracy': result_e4['best_val_accuracy'],
    'best_val_loss': result_e4['best_val_loss']
})
print(f"E4 Val Acc: {result_e4['best_val_accuracy']:.4f}, Test Acc: {result_e4['test_accuracy']:.4f}")

os.makedirs('artifacts/figures', exist_ok=True)
torch.save(result_e4['model'].state_dict(), 'artifacts/best_model.pt')

best_config_json = {
    'architecture': {'hidden_sizes': best_config_a['hidden_sizes'], 'num_classes': 47,
                     'dropout_p': best_config_a['dropout_p'], 'use_batchnorm': best_config_a['use_batchnorm']},
    'optimizer': 'Adam', 'lr': 1e-3, 'seed': SEED, 'dataset': 'EMNIST_balanced',
    'early_stopping': {'patience': 5}
}
with open('artifacts/best_config.json', 'w') as f:
    json.dump(best_config_json, f, indent=2)

save_curves(result_e4['history'], 'E4 (Best Model)', 'artifacts/figures/curves_best.png')
print("Saved best_model.pt, best_config.json, curves_best.png")

arch_config = best_config_a.copy()

print("\n=== O1: LR too large (1e-1) ===")
config_o1 = arch_config.copy()
config_o1['lr'] = 1e-1
result_o1 = run_experiment(config_o1, train_loader, val_loader, test_loader, device, epochs=8)
results.append({
    'experiment_id': 'O1', 'dataset': 'EMNIST', 'seed': SEED,
    'model_summary': '256-128-64/ReLU/reg-as-E4',
    'optimizer': 'Adam', 'lr': 1e-1, 'momentum': 0, 'weight_decay': 0,
    'epochs_trained': result_o1['epochs_trained'],
    'best_val_accuracy': result_o1['best_val_accuracy'],
    'best_val_loss': result_o1['best_val_loss']
})
print(f"O1 Val Acc: {result_o1['best_val_accuracy']:.4f} (expected: unstable)")

print("\n=== O2: LR too small (1e-5) ===")
config_o2 = arch_config.copy()
config_o2['lr'] = 1e-5
result_o2 = run_experiment(config_o2, train_loader, val_loader, test_loader, device, epochs=8)
results.append({
    'experiment_id': 'O2', 'dataset': 'EMNIST', 'seed': SEED,
    'model_summary': '256-128-64/ReLU/reg-as-E4',
    'optimizer': 'Adam', 'lr': 1e-5, 'momentum': 0, 'weight_decay': 0,
    'epochs_trained': result_o2['epochs_trained'],
    'best_val_accuracy': result_o2['best_val_accuracy'],
    'best_val_loss': result_o2['best_val_loss']
})
print(f"O2 Val Acc: {result_o2['best_val_accuracy']:.4f} (expected: slow convergence)")

print("\n=== O3: SGD+momentum + weight_decay ===")
config_o3 = arch_config.copy()
config_o3['optimizer'] = 'SGD'
config_o3['lr'] = 1e-2
config_o3['momentum'] = 0.9
config_o3['weight_decay'] = 1e-4
result_o3 = run_experiment(config_o3, train_loader, val_loader, test_loader, device, epochs=15)
results.append({
    'experiment_id': 'O3', 'dataset': 'EMNIST', 'seed': SEED,
    'model_summary': '256-128-64/ReLU/reg-as-E4',
    'optimizer': 'SGD', 'lr': 1e-2, 'momentum': 0.9, 'weight_decay': 1e-4,
    'epochs_trained': result_o3['epochs_trained'],
    'best_val_accuracy': result_o3['best_val_accuracy'],
    'best_val_loss': result_o3['best_val_loss']
})
print(f"O3 Val Acc: {result_o3['best_val_accuracy']:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(result_o1['history']['train_loss'], label='O1 Train Loss (LR=1e-1)', color='red', alpha=0.7)
ax1.plot(result_o1['history']['val_loss'], label='O1 Val Loss', color='red', linestyle='--', alpha=0.7)
ax1.plot(result_o2['history']['train_loss'], label='O2 Train Loss (LR=1e-5)', color='blue', alpha=0.7)
ax1.plot(result_o2['history']['val_loss'], label='O2 Val Loss', color='blue', linestyle='--', alpha=0.7)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('LR Extremes: Loss Comparison')
ax1.legend(fontsize=8)
ax1.grid(True)

ax2.plot(result_o1['history']['train_acc'], label='O1 Train Acc', color='red', alpha=0.7)
ax2.plot(result_o1['history']['val_acc'], label='O1 Val Acc', color='red', linestyle='--', alpha=0.7)
ax2.plot(result_o2['history']['train_acc'], label='O2 Train Acc', color='blue', alpha=0.7)
ax2.plot(result_o2['history']['val_acc'], label='O2 Val Acc', color='blue', linestyle='--', alpha=0.7)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('LR Extremes: Accuracy Comparison')
ax2.legend(fontsize=8)
ax2.grid(True)

plt.tight_layout()
plt.savefig('artifacts/figures/curves_lr_extremes.png', dpi=150)
plt.close()
print(" Saved curves_lr_extremes.png")


df_results = pd.DataFrame(results)
df_results.to_csv('artifacts/runs.csv', index=False)
print("\n Results summary:")
print(df_results[['experiment_id', 'best_val_accuracy', 'epochs_trained']].to_string(index=False))

print(f"\n Final test accuracy (E4 best model): {result_e4['test_accuracy']:.4f}")

Using device: cpu


100.0%


Train batches: 705, Val batches: 177, Test batches: 147
Batch shape: x=torch.Size([128, 1, 28, 28]), y=torch.Size([128])
Value range: [-0.424, 2.821]
Classes: 45 unique in batch, total: 47

=== E1: Base MLP ===
E1 Val Acc: 0.8397

=== E2: + Dropout (p=0.3) ===
E2 Val Acc: 0.8347

=== E3: + BatchNorm ===
E3 Val Acc: 0.8519

=== E4: EarlyStopping ===
Early stopping at epoch 15
E4 Val Acc: 0.8519, Test Acc: 0.8500
Saved best_model.pt, best_config.json, curves_best.png

=== O1: LR too large (1e-1) ===
O1 Val Acc: 0.8120 (expected: unstable)

=== O2: LR too small (1e-5) ===
O2 Val Acc: 0.6539 (expected: slow convergence)

=== O3: SGD+momentum + weight_decay ===
O3 Val Acc: 0.8505
 Saved curves_lr_extremes.png

 Results summary:
experiment_id  best_val_accuracy  epochs_trained
           E1           0.839672              15
           E2           0.834663              15
           E3           0.851906              15
           E4           0.851906              15
           O1         